# Silver Claims Layer — Overview

## Goal
Create:
- silver_claims
- quarantine_invalid_claims

## Purpose
This notebook validates insurance claims data:
- ensures correct IDs
- checks relationships with policies & customers
- applies business rules
- separates invalid records

## Output
- Trusted claims data (Silver)
- Rejected claims (Quarantine)

In [0]:
from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"

valid_status = ["open", "approved", "rejected", "under_review", "paid"]

In [0]:
claims_bronze = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_claims")

policies = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_policies").select("policy_id").dropDuplicates()
customers = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers").select("customer_id").dropDuplicates()

In [0]:
claims_prepared = (
    claims_bronze
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("policy_id", F.trim(F.col("policy_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("claim_type", F.lower(F.trim(F.col("claim_type"))))
    .withColumn("claim_status", F.lower(F.trim(F.col("claim_status"))))
    .withColumn("reported_channel", F.lower(F.trim(F.col("reported_channel"))))
    .withColumn("claim_amount", F.col("claim_amount").cast("double"))
    .withColumn("claim_date", F.to_date(F.col("claim_date")))
    .withColumn("created_at", F.to_timestamp(F.col("created_at")))
)

In [0]:
claims_joined = (
    claims_prepared.alias("cl")
    .join(policies.alias("p"), "policy_id", "left")
    .join(customers.alias("c"), "customer_id", "left")
    .withColumn("policy_exists", F.col("p.policy_id").isNotNull())
    .withColumn("customer_exists", F.col("c.customer_id").isNotNull())
    .select("cl.*", "policy_exists", "customer_exists")
)

In [0]:
invalid_claims = (
    claims_joined
    .filter(
        F.col("claim_id").isNull()
        | F.col("policy_id").isNull()
        | F.col("customer_id").isNull()
        | (~F.col("policy_exists"))
        | (~F.col("customer_exists"))
        | (~F.col("claim_status").isin(valid_status))
        | (F.col("claim_amount") <= 0)
    )
    .withColumn("record_id", F.col("claim_id"))
    .withColumn("source_table", F.lit("bronze_claims"))
    .withColumn("error_reason", F.lit("claim_quality_rule_failed"))
    .withColumn("error_severity", F.lit("HIGH"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn(
        "original_record_json",
        F.to_json(F.struct(*[F.col(c) for c in claims_prepared.columns]))
    )
)

In [0]:
silver_claims = (
    claims_joined
    .filter(
        F.col("claim_id").isNotNull()
        & F.col("policy_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("policy_exists")
        & F.col("customer_exists")
        & F.col("claim_status").isin(valid_status)
        & (F.col("claim_amount") > 0)
    )
    .drop("policy_exists", "customer_exists")
    .dropDuplicates(["claim_id"])
)

In [0]:
silver_claims.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims")

In [0]:
(
    invalid_claims
    .select(
        "record_id",
        "source_table",
        "error_reason",
        "error_severity",
        "quarantine_timestamp",
        "source_file_name",
        "ingest_run_id",
        "original_record_json"
    )
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_claims")
)

In [0]:
print("Bronze claims:", claims_bronze.count())
print("Silver claims:", spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims").count())
print("Quarantine claims:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_claims").count())

In [0]:
%sql
SELECT * 
FROM insurance_lakehouse.silver.silver_claims 
LIMIT 5;